# Viscous drag: a small steel ball in glycerine — Year 3

Simulation with the **step-by-step method** (numpy and matplotlib). VPython does not work in Google Colab: for the 3D animation use the script on https://glowscript.org.

Forces on the ball ($y$ axis pointing down): weight $mg$, buoyancy $\rho_f V g$ upwards, viscous drag $b\,v$ upwards, with $b = 6\pi\eta r$ (Stokes' law).
$$m\,a = mg - \rho_f V g - b\,v$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# data (SI)
g = 9.81          # m/s^2
r = 2.0e-3        # radius of the ball (m)
rho_s = 7800      # steel (kg/m^3)
rho_f = 1260      # glycerine (kg/m^3)
eta = 1.4         # viscosity of glycerine (Pa·s)

V = 4/3*np.pi*r**3
m = rho_s*V
b = 6*np.pi*eta*r

a0 = g*(1 - rho_f/rho_s)
vL = (m - rho_f*V)*g/b
tau = m/b
Re = rho_f*vL*2*r/eta
print(f"m = {m*1000:.3f} g,  b = {b:.4f} kg/s")
print(f"a0 = {a0:.2f} m/s^2,  vL = {vL*100:.2f} cm/s,  tau = {tau*1000:.2f} ms,  Re = {Re:.2f}")

## Step-by-step method
At each step: compute the acceleration with the current velocity, then update the velocity: $v_{n+1} = v_n + a_n\,\Delta t$.

In [ ]:
def step_by_step(dt, t_end=0.03):
    n = int(round(t_end/dt))
    t = np.zeros(n+1); v = np.zeros(n+1); a = np.zeros(n+1)
    for i in range(n):
        a[i] = (m*g - rho_f*V*g - b*v[i])/m
        v[i+1] = v[i] + a[i]*dt
        t[i+1] = t[i] + dt
    a[n] = (m*g - rho_f*V*g - b*v[n])/m
    return t, v, a

# table with dt = 1 ms (the same as in the lesson summary)
t1, v1, a1 = step_by_step(1e-3, 0.007)
print(" n  t(ms)  v(cm/s)  a(m/s^2)")
for i in range(len(t1)):
    print(f"{i:2d}  {t1[i]*1000:4.0f}   {v1[i]*100:6.2f}   {a1[i]:6.2f}")

In [ ]:
t, v, a = step_by_step(1e-4)        # dt = 0.1 ms
t_exact = np.linspace(0, 0.03, 300)
v_exact = vL*(1 - np.exp(-t_exact/tau))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t*1000, v*100, label="step by step (dt = 0.1 ms)")
ax[0].plot(t_exact*1000, v_exact*100, "--", label="v_L(1 - e^(-t/τ))")
ax[0].plot(t1*1000, v1*100, "o", label="dt = 1 ms")
ax[0].axhline(vL*100, color="gray", lw=0.8)
ax[0].set_xlabel("t (ms)"); ax[0].set_ylabel("v (cm/s)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(t*1000, a)
ax[1].axhline(g, color="red", ls="--", label="g (free fall in a vacuum)")
ax[1].set_xlabel("t (ms)"); ax[1].set_ylabel("a (m/s²)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

i63 = np.argmax(v >= 0.632*vL)
print(f"v reaches 63% of vL after {t[i63]*1000:.2f} ms (theoretical tau = {tau*1000:.2f} ms)")
print(f"v after 30 ms = {v[-1]*100:.3f} cm/s (vL = {vL*100:.3f} cm/s)")

## Experiment: $v_L$ as a function of the radius
If $v_L \propto r^2$, the graph of $v_L$ against $r^2$ is a straight line through the origin.

In [ ]:
radii = np.array([1.0, 1.5, 2.0, 2.5])*1e-3
vL_r = 2*radii**2*(rho_s - rho_f)*g/(9*eta)
for rr, vv in zip(radii, vL_r):
    print(f"r = {rr*1000:.1f} mm   r^2 = {(rr*1000)**2:.2f} mm^2   vL = {vv*100:.2f} cm/s   Re = {rho_f*vv*2*rr/eta:.2f}")
plt.plot((radii*1000)**2, vL_r*100, "o-")
plt.xlabel("r² (mm²)"); plt.ylabel("v_L (cm/s)"); plt.grid(alpha=0.3); plt.show()

## Challenge: the same ball in water
Stokes' formula gives a huge terminal velocity. Is the model still valid? Check the Reynolds number and compare with quadratic drag $\tfrac12 C\rho_f A v^2$ ($C = 0.4$).

In [ ]:
rho_water, eta_water, C = 1000, 1.0e-3, 0.4
vL_stokes = 2*r**2*(rho_s - rho_water)*g/(9*eta_water)
Re_stokes = rho_water*vL_stokes*2*r/eta_water
A = np.pi*r**2
vL_quad = np.sqrt((m - rho_water*V)*g/(0.5*C*rho_water*A))
Re_quad = rho_water*vL_quad*2*r/eta_water
print(f"Stokes:    vL = {vL_stokes:.1f} m/s,  Re = {Re_stokes:.1e}  -> Stokes does NOT hold")
print(f"quadratic: vL = {vL_quad:.2f} m/s,  Re = {Re_quad:.0f}  -> turbulent regime, consistent model")